# Feature Engineering — FIFA World Cup 2026 Player Performance

No `01_eda.ipynb` exploramos os dados e tomamos algumas decisões importantes:

- **Alvo:** regressão em `player_rating`.
- **Escopo:** filtrar para `minutes_played > 0` — 42% das linhas são jogadores que não entraram em campo, onde a nota é trivialmente 0.
- **Vazamento de dados confirmado:** `performance_score` (e, por extensão, `tournament_rating`, quase idêntico ao alvo) ficam fora das features.
- **Falso alarme:** `top_speed_kmh` parecia vazamento (corr. 0.98) mas era um artefato das linhas sem minutos jogados — não tem sinal real, fica de fora por ser inútil, não por vazar informação.
- **Ruído sem confiabilidade:** os totais acumulados de torneio (`total_goals_tournament` etc.) não têm coerência temporal — ficam de fora.

**Objetivo deste notebook:** partir dessas decisões, montar de fato o conjunto de features (selecionar colunas, tratar categóricas de alta cardinalidade) e deixar os dados prontos para a etapa de modelagem.

## 1. Importando as ferramentas

Começamos só com pandas, seaborn e matplotlib — as mesmas do notebook anterior, pra manipular os dados e visualizar as features conforme formos construindo. Ferramentas específicas de encoding (scikit-learn) a gente importa mais adiante, no momento em que forem realmente usadas.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Carregando os dados e aplicando o escopo

Cada notebook parte do CSV original (não temos módulos `.py` compartilhados de propósito — o projeto é só notebooks). A primeira coisa a fazer é reaplicar a decisão de escopo do notebook anterior: **remover as linhas de jogadores que não entraram em campo** (`minutes_played == 0`), já que prever nota 0 pra quem não jogou é trivial e não é o problema que queremos resolver.

In [2]:
data_df = pd.read_csv("../data/fifa_world_cup_2026_player_performance.csv")

played_df = data_df[data_df["minutes_played"] > 0].copy()

print(f"Linhas antes do filtro: {len(data_df)}")
print(f"Linhas depois do filtro (minutes_played > 0): {len(played_df)}")

Linhas antes do filtro: 54600
Linhas depois do filtro (minutes_played > 0): 31558


## 3. Selecionando as colunas candidatas

Com base na conclusão do `01_eda`, montamos a lista do que **sai** do conjunto de features:

- Identificadores: `player_id`, `player_name`, `match_id`
- Vazamento: `performance_score`, `tournament_rating`
- Sem sinal real: `top_speed_kmh`
- Ruído sem coerência temporal: `total_goals_tournament`, `total_assists_tournament`, `total_minutes_tournament`
- Outro resultado pós-partida (não uma causa do rating): `player_of_match_awards`
- O próprio alvo: `player_rating` (guardamos à parte, não é uma feature)

Tudo o que sobra é candidato a feature.

In [3]:
excluded_cols = [
    # identificadores
    "player_id", "player_name", "match_id",
    # vazamento de dados
    "performance_score", "tournament_rating",
    # sem sinal real (artefato investigado no notebook anterior)
    "top_speed_kmh",
    # ruído sem coerência temporal
    "total_goals_tournament", "total_assists_tournament", "total_minutes_tournament",
    # outro resultado pós-partida, não uma causa do rating
    "player_of_match_awards",
]

target_col = "player_rating"

feature_cols = [c for c in played_df.columns if c not in excluded_cols + [target_col]]

print(f"Features candidatas: {len(feature_cols)}")
feature_cols

Features candidatas: 64


['age',
 'nationality',
 'team',
 'jersey_number',
 'position',
 'height_cm',
 'weight_kg',
 'preferred_foot',
 'club_name',
 'market_value_eur',
 'match_date',
 'stadium',
 'city',
 'opponent_team',
 'tournament_stage',
 'match_result',
 'goals_team',
 'goals_opponent',
 'minutes_played',
 'goals',
 'assists',
 'shots',
 'shots_on_target',
 'expected_goals_xg',
 'expected_assists_xa',
 'key_passes',
 'successful_passes',
 'total_passes',
 'pass_accuracy',
 'dribbles_attempted',
 'successful_dribbles',
 'crosses',
 'successful_crosses',
 'tackles',
 'interceptions',
 'clearances',
 'blocks',
 'aerial_duels_won',
 'aerial_duels_lost',
 'recoveries',
 'defensive_actions',
 'fouls_committed',
 'fouls_suffered',
 'yellow_cards',
 'red_cards',
 'offsides',
 'saves',
 'save_percentage',
 'punches',
 'clean_sheet',
 'goals_conceded',
 'penalty_saves',
 'distance_covered_km',
 'sprint_distance_km',
 'accelerations',
 'decelerations',
 'stamina_score',
 'offensive_contribution',
 'defensive_con

## 4. Cardinalidade das colunas categóricas

Modelos de ML não entendem texto — colunas categóricas (`position`, `team`, etc.) precisam virar números antes de entrar no modelo. A técnica mais simples é o **one-hot encoding** (uma coluna binária por categoria), mas ela só funciona bem quando a coluna tem poucas categorias — com muitas, o número de colunas explode e a maioria fica quase sempre zero.

Por isso, antes de decidir a técnica, precisamos saber: quantas categorias cada coluna tem?

In [4]:
categorical_cols = played_df[feature_cols].select_dtypes(include="str").columns.tolist()

played_df[categorical_cols].nunique().sort_values()

preferred_foot        2
match_result          3
position              4
tournament_stage      7
stadium              16
city                 16
nationality          48
team                 48
opponent_team        48
match_date           51
club_name           122
dtype: int64

Dá pra dividir em três grupos:

- **Baixa cardinalidade (2 a 16 categorias):** `preferred_foot`, `match_result`, `position`, `tournament_stage`, `stadium`, `city` — one-hot encoding direto, sem problema.
- **Cardinalidade média (48):** `nationality`, `team`, `opponent_team` — one-hot ainda é viável (48 colunas novas não é absurdo com 31.558 linhas), mas vale checar redundância antes.
- **Alta cardinalidade:** `club_name` (122) e `match_date` (51) — one-hot aqui geraria muitas colunas esparsas; precisam de uma abordagem diferente.

Antes de decidir a técnica, um detalhe chamou atenção: `nationality` e `team` têm exatamente a mesma cardinalidade (48). Pode ser coincidência, ou pode ser a mesma informação escrita de duas formas (ex: "Spanish" vs "Spain"). Vale checar.

In [5]:
nationality_team_map = played_df[["nationality", "team"]].drop_duplicates()

print(f"Combinações únicas de (nationality, team): {len(nationality_team_map)}")
nationality_team_map.head()

Combinações únicas de (nationality, team): 48


,nationality,team
0,Spanish,Spain
26,South African,South Africa
52,Peruvian,Peru
78,Dutch,Netherlands
104,Moroccan,Morocco


Confirmado: 48 combinações únicas para 48 categorias em cada coluna — é uma relação 1 para 1. `nationality` é só o gentílico de `team` ("Spanish" → "Spain"). São a mesma informação escrita de duas formas; `nationality` sai do conjunto de features (redundante com `team`).